# 🧵 Multithreading en Python

## ❓ Python peut-il faire du multithreading ?

Réponse courte : **oui... mais** cela dépend de ce que l'on entend par "multithreading". Clarifions d'abord les notions clés.

## 📦 Qu'est-ce qu'un processus ?

Un **processus** est une instance d'un programme en cours d'exécution.

Quand vous lancez une application (navigateur, script Python, éditeur), le système d'exploitation crée un processus. Chaque processus possède :

* son propre espace mémoire,
* ses propres variables et données,
* ses propres ressources (fichiers, connexions réseau, etc.).

Les processus sont **isolés** les uns des autres par l'OS. Un processus n'accède pas directement à la mémoire d'un autre. Cet isolement améliore stabilité et sécurité, mais la communication inter-processus est plus coûteuse.

Pour lister les processus en cours, utilisez par exemple `ps` (Linux/Mac) ou le Gestionnaire des tâches (Windows).

## 🧶 Qu'est-ce qu'un thread ?

Un **thread** est une unité d'exécution plus petite *à l'intérieur* d'un processus.

Un processus peut contenir :

* un thread (processus mono-thread),
* plusieurs threads (processus multi-thread).

Les threads d'un même processus :

* partagent le même espace mémoire,
* partagent variables et données,
* communiquent très efficacement.

Comme la mémoire est partagée, il faut cependant coordonner les accès pour éviter les conditions de course et la corruption des données.

Exemple d'un processus pour un onglet Chrome :

```
Process: Chrome tab
├─ UI thread
├─ JavaScript engine thread
├─ Network thread
├─ Rendering thread
```

Autre exemple avec un éditeur de texte :

```
Process: Text Editor
├─ Main thread → UI event loop
├─ Auto-save thread
├─ Spell-check thread
```

## 🎯 Applications : I/O-bound vs CPU-bound

* **Applications I/O-bound** : passent l'essentiel du temps à attendre des entrées/sorties (disque, réseau, utilisateur). Exemples : serveurs web, traitement de fichiers, requêtes base de données.
* **Applications CPU-bound** : passent l'essentiel du temps à calculer. Exemples : calcul scientifique, traitement d'image, analyse de données.

## ⚖️ Multiprocessing vs Multithreading

### 🔀 Multiprocessing

Le **multiprocessing** consiste à exécuter plusieurs processus en parallèle.

Caractéristiques :

* mémoire séparée,
* consommation mémoire plus élevée,
* communication plus coûteuse,
* isolation et robustesse fortes.

Cas d'usage :

* tâches CPU-bound,
* calcul parallèle,
* charges exploitant plusieurs cœurs CPU.

En Python, cela signifie plusieurs interpréteurs Python indépendants, chacun dans son propre processus.

### 🧵 Multithreading

Le **multithreading** consiste à exécuter plusieurs threads au sein d'un même processus.

Caractéristiques :

* mémoire partagée,
* faible coût de communication,
* changements de contexte rapides,
* risque plus élevé de bugs de concurrence.

Cas d'usage :

* tâches I/O-bound (fichiers, réseau, base),
* applications réactives (GUI, serveurs).

En Python, cela signifie plusieurs threads dans le même processus d'interpréteur Python.

### 🔑 Différence conceptuelle

| Aspect          | Multithreading         | Multiprocessing |
| --------------- | ---------------------- | --------------- |
| Mémoire         | Partagée               | Isolée          |
| Communication   | Rapide                 | Plus lente      |
| Sécurité        | Risque de race         | Plus sûr         |
| Parallélisme CPU| Limité en Python       | Complet         |

## 💻 Cœurs CPU et parallélisme matériel

Un **cœur CPU** est une unité physique d'exécution capable d'exécuter des instructions de manière indépendante.

Les CPU modernes ont généralement :

* plusieurs cœurs physiques,
* parfois plusieurs *threads matériels* par cœur (ex. Hyper-Threading).

Chaque cœur exécute un flux d'instructions à la fois. Avec plusieurs cœurs, plusieurs tâches peuvent réellement s'exécuter simultanément.

## 📅 Petit rappel historique : avant/après 2005

Avant environ **2005**, la plupart des processeurs grand public avaient :

* un seul cœur,
* des gains de performance surtout via l'augmentation de fréquence.

Le pseudo-parallélisme venait surtout du **time slicing** (alternance rapide des tâches).

Vers 2005, les limites thermiques et énergétiques ont freiné la hausse des fréquences. L'industrie est passée à :

* des processeurs multi-cœurs,
* du parallélisme matériel.

Depuis, les logiciels doivent davantage intégrer concurrence et parallélisme.

## 🔄 Concurrence vs parallélisme

Ces notions sont proches mais différentes :

* **Concurrence** : plusieurs tâches progressent sur la même période.
* **Parallélisme** : plusieurs tâches s'exécutent en même temps sur des cœurs différents.

Un CPU monocœur peut gérer de la concurrence. Un CPU multicœur permet le parallélisme réel.

## 🔒 Le Global Interpreter Lock (GIL)

### 🤔 Qu'est-ce que le GIL ?

Quand Python a été créé (fin des années 1980), les architectures étaient surtout monocœur. Pour simplifier la gestion mémoire et la sûreté des threads, les créateurs ont introduit le **Global Interpreter Lock (GIL)** en 1992.

Le GIL empêche plusieurs threads natifs d'exécuter en même temps le bytecode Python. Donc, même avec plusieurs threads, un seul exécute du code Python à un instant donné.

Cela peut sembler une limite majeure face à des langages qui gèrent mieux le multithreading natif (C++, Rust, Java...). Le sujet a longtemps fait débat.

Cependant, comme l'a rappelé Larry Hastings (core developer), le GIL est aussi l'une des raisons de la popularité de Python.

Le **GIL** est un mutex (verrou d'exclusion mutuelle) propre à **CPython** (implémentation de référence).

Règle :

> Un seul **thread** peut exécuter du bytecode Python à la fois *par processus*.

Donc, même sur machine multicœur, les threads Python d'un même processus n'exécutent pas du code Python en parallèle.

<img src='files/gil.png' alt='GIL diagram' width='600'>

### ❓ Pourquoi existe-t-il ?

Le GIL simplifie :

* la gestion mémoire,
* le ramasse-miettes (garbage collection),
* la sûreté des objets Python face aux threads.

Résultat : beaucoup de programmes mono-thread Python sont **plus simples et souvent plus rapides** qu'ils ne le seraient sans GIL.

### ✅ Ce que le GIL n'empêche pas

* Les threads restent concurrents.
* Les threads peuvent relâcher le GIL pendant les I/O bloquantes.
* Les extensions natives en C peuvent relâcher le GIL.

C'est pourquoi le multithreading Python reste très utile pour :

* les tâches I/O-bound,
* les serveurs réseau,
* les workloads avec beaucoup d'attente.

### 📚 Références sur le GIL

* ["It isn't Easy to Remove the GIL" par Guido van Rossum (2007)](https://www.artima.com/weblogs/viewpost.jsp?thread=214235)
* [Larry Hastings sur le GIL à PyCon (2015)](https://www.youtube.com/watch?v=KVKufdTphKs)
* [Interview de Guido van Rossum sur le GIL (2022)](https://www.youtube.com/watch?v=m4zDBk0zAUY)

### 🎯 Alors, Python est multithreadé ou pas ?

**>>>** Techniquement oui, mais pas "multithreadé simultanément" pour l'exécution du bytecode Python dans un même processus CPython.

## 📦 Le module threading

Python fournit le module intégré `threading` pour créer et gérer des threads.

In [ ]:
import threading
import time

### 🧩 L'objet `Thread`

En Python, un thread est représenté par la classe `Thread` du module `threading`. Vous pouvez créer un nouveau thread en instanciant cette classe et en lui passant une fonction cible à exécuter dans ce thread.

In [ ]:
# A function to be run in a thread
def print_number():
    for i in range(5):
        print(f"Hello from the thread! number is {i}")
        time.sleep(0.5)

# Main thread execution
thread = threading.Thread(target=print_number)
thread.start()
print("Hello from the main thread!")

Comme vous le voyez dans l'exemple ci-dessus, la classe `Thread` permet de lancer une fonction en parallèle du thread principal.

Mais un détail pose problème : l'affichage du thread principal arrive avant la fin du nouveau thread.

Corrigeons cela.

### 🔗 Utiliser la méthode `.join()`

La méthode `join()` permet au thread principal d'attendre qu'un thread se termine avant de continuer. Cela garantit que le programme principal ne se termine pas prématurément.

In [ ]:
# A function to be run in a thread
def print_number():
    for i in range(5):
        print(f"Hello from the thread! number is {i}")
        time.sleep(0.5)

# Main thread execution
thread = threading.Thread(target=print_number)
thread.start()
thread.join()  # Wait for the thread to finish
print("Hello from the main thread!")

### 📥 Utiliser des threads avec des arguments

In [ ]:
# A function to be run in a thread
def print_number(message, count):
    for i in range(count):
        print(f"{message} number is {i}")
        time.sleep(0.5)

# Main thread execution
thread = threading.Thread(target=print_number,
                          args=("Hello from the thread!", 5))
thread.start()
thread.join()  # Wait for the thread to finish
print("Hello from the main thread!")

### 🔢 Créer plusieurs threads en une fois

In [ ]:
# A function to be run in a thread
def print_number(message, count):
    for i in range(count):
        print(f"{message} number is {i}")
        time.sleep(0.5)

# Main thread execution
threads = []
for i in range(3):
    thread = threading.Thread(target=print_number,
                              args=(f"{' ' * i}Thread-{i+1}", 5))
    threads.append(thread)
    thread.start()
    # thread.join()  # this would make threads run sequentially

for thread in threads:
    thread.join()  # Wait for all threads to finish
print("Hello from the main thread!")

### 👻 Threads daemon (démon)

Un thread daemon est un thread d'arrière-plan qui n'empêche pas le programme de se terminer. Quand le programme principal s'arrête, tous les threads daemon sont automatiquement arrêtés.

Attention : l'exemple suivant fonctionne dans un fichier `.py`, mais pas dans un notebook Jupyter, car le noyau ne s'arrête pas après l'exécution d'une cellule. Le daemon continue donc de tourner en arrière-plan.

### ⚠️ Conditions de course (race conditions)

Ici, l'expression `time.sleep(0)` relâche le GIL et indique à l'ordonnanceur : "change de thread si tu veux".

In [ ]:
import threading
import time

counter = 0

def increment():
    global counter
    for _ in range(10_000):
        tmp = counter
        time.sleep(0)  # force a context switch
        counter = tmp + 1

threads = [threading.Thread(target=increment) for _ in range(4)]

for t in threads:
    t.start()
for t in threads:
    t.join()

print(counter)

La sortie devrait être 40 000, mais on obtient plutôt 10 000, 10 001 ou 10 002...

Pourquoi ? Parce que tous les threads accèdent et modifient la variable partagée `counter` en même temps, sans mécanisme de synchronisation. La plupart des incréments sont donc perdus, et quelques-uns passent selon le timing.

Alors pourquoi le GIL ne nous protège-t-il pas ici ? Parce qu'il n'empêche pas les conditions de course. Le GIL garantit seulement qu'un seul thread exécute du bytecode Python à la fois, mais la condition de course se produit entre plusieurs instructions bytecode.

### 🔐 Utiliser `threading.Lock()`

Pour corriger la condition de course, on peut utiliser un `Lock` du module `threading`. Un verrou est une primitive de synchronisation qui garantit qu'un seul thread à la fois accède à une ressource partagée.

In [ ]:
import threading
import time

counter = 0
counter_lock = threading.Lock() # create a lock object

def increment():
    global counter
    with counter_lock: # acquire the lock
        for _ in range(10_000):
            tmp = counter
            time.sleep(0)
            counter = tmp + 1

threads = [threading.Thread(target=increment) for _ in range(4)]

for t in threads:
    t.start()
for t in threads:
    t.join()

print(counter)

### 🎯 Le module `concurrent.futures`

Le module `concurrent.futures` fournit une interface de haut niveau pour exécuter des appels de manière asynchrone.
Il propose deux classes principales :

* `ThreadPoolExecutor` : gestion d'un pool de threads
* `ProcessPoolExecutor` : gestion d'un pool de processus

#### 🤔 Qu'est-ce qu'un exécuteur ?

Un **exécuteur** est un objet qui gère un pool de threads ou de processus et offre des méthodes pour soumettre des tâches à exécuter.

Il masque les détails bas niveau de gestion de concurrence, ce qui permet de se concentrer sur les tâches à lancer en parallèle.

#### ⚖️ Différence entre ThreadPoolExecutor et ProcessPoolExecutor

* `ThreadPoolExecutor` est adapté aux tâches I/O-bound où les threads sont utiles malgré le GIL, car ils peuvent relâcher le GIL pendant les opérations bloquantes.
* `ProcessPoolExecutor` est adapté aux tâches CPU-bound où un vrai parallélisme est nécessaire, car chaque processus possède son propre interpréteur Python et son propre espace mémoire, contournant le GIL (plus de détails au chapitre suivant).

In [ ]:
# With ThreadPoolExecutor

import threading
import concurrent.futures
import time

def task(n):
    print(f"{" " * n}Task {n} starting on thread {threading.current_thread().name}")
    time.sleep(n)
    print(f" {" " * n}Task {n} completed on thread {threading.current_thread().name}")
    return n * n

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(task, [1, 2, 3, 4, 5])

print("Results:", list(results))
print("Main thread finished.")

En observant les sorties, on voit que l'exécuteur décide lui-même quelle tâche exécuter sur quel thread ou processus.

### 🚦 Événements de thread

Un `threading.Event` est une primitive de synchronisation qui permet aux threads de communiquer en se signalant des événements.
Un événement peut être dans deux états : "set" (activé) ou "clear" (non activé). Les threads peuvent attendre qu'un événement soit activé avant de poursuivre.

Imaginons donc un thread que l'on veut démarrer à un moment très précis.

In [ ]:
def worker(event):
    print("    Worker is waiting for the event to be set.")
    event.wait()  # Wait until the event is set
    print("    Worker has detected the event is set and is proceeding.")

    for _ in range(5):
        print("    Worker is working...")
        time.sleep(1)
    print("    Worker has finished its work.")
    
event = threading.Event()
thread = threading.Thread(target=worker, args=(event,))
thread.start()
time.sleep(3)  # Simulate some setup time in the main thread
print("Main thread is setting the event.")
event.set()  # Signal the event
thread.join()
print("Main thread finished.")

### 💪 Exercice

Le code suivant récupère des données depuis différents sites web. Modifiez la fonction `run_fetch_url` et utilisez le multithreading pour récupérer les données en parallèle afin d'améliorer le temps d'exécution global.

Vous pouvez :

- Utiliser `threading` pour créer et gérer les threads manuellement.
- Utiliser `concurrent.futures.ThreadPoolExecutor` pour une approche plus haut niveau.
- (Ou faire les deux !)

In [ ]:
import requests
import time

def fetch_url(url):
    response = requests.get(url)
    print(f"""Fetched {url} with status code {response.status_code}.
            Data is {len(response.content)} bytes.""")

def run_fetch_url(urls):
    start_time = time.time()
    for url in urls:fetch_url(url)
    end_time = time.time()
    print(f"Total time taken: {end_time - start_time} seconds")

urls = ['https://stackoverflow.com',
        'https://github.com',
        'https://python.org',
        'https://wikipedia.org',
        'https://reddit.com',
        'https://google.com',
        'https://youtube.com',]

run_fetch_url(urls)

In [ ]:
# Code the new function here!

